## <font style="color:blue">Project 4: Kaggle Competition - Semantic Segmentation</font>

#### Maximum Points: 100

<div>
    <table>
        <tr><td><h6>Sr. no.</h6></td> <td><h6>Section</h6></td> <td><h6>Points</h6></td> </tr>
        <tr><td><h6>1</h6></td> <td><h6>1.1. Dataset Class</h6></td> <td><h6>7</h6></td> </tr>
        <tr><td><h6>2</h6></td> <td><h6>1.2. Visualize dataset</h6></td> <td><h6>3</h6></td> </tr>
        <tr><td><h6>3</h6></td> <td><h6>2. Evaluation Metrics</h6></td> <td><h6>10</h6></td> </tr>
        <tr><td><h6>4</h6></td> <td><h6>3. Model</h6></td> <td><h6>10</h6></td> </tr>
        <tr><td><h6>5</h6></td> <td><h6>4.1. Train</h6></td> <td><h6>7</h6></td> </tr>
        <tr><td><h6>6</h6></td> <td><h6>4.2. Inference</h6></td> <td><h6>3</h6></td> </tr>
        <tr><td><h6>7</h6></td> <td><h6>5. Prepare Submission CSV</h6></td><td><h6>10</h6></td> </tr>
        <tr><td><h6>8</h6></td> <td><h6>6. Kaggle Profile Link</h6></td> <td><h6>50</h6></td> </tr>
    </table>
</div>

---

<h2>Dataset Description </h2>
<p>The dataset consists of 3,269 images in 12 classes (including background). All images were taken from drones in a variety of scales. Samples are shown below:
<img src="https://github.com/ishann/aeroscapes/blob/master/assets/data_montage.png?raw=true" width="800" height="800">
<p>The data was splitted into public train set and private test set which is used for evaluation of submissions.

### Prepare folder structure, download data, etc

#### Cloud computer

In [1]:
# !mkdir -p /workspace/input/data
# !mkdir -p /workspace/semantic_segmentation
# !mkdir -p /workspace/output
# !apt update && apt install -y unzip
# !wget --directory-prefix /workspace/input/data -nc https://public-bucket-with-cards.s3.us-east-1.amazonaws.com/opencv-pytorch-segmentation-project.zip
# !unzip -d /workspace/input/data -nq /workspace/input/data/opencv-pytorch-segmentation-project.zip 

#### Kaggle

In [2]:
# !cp -r /kaggle/input/semantic-segmentation/* /kaggle/working/

### Install dependencies

In [3]:
# !pip install -q -r requirements.txt
# !pip uninstall -q -y opencv-python
# !pip install -q --force-reinstall opencv-python-headless numpy==1.26.*

In [4]:
%load_ext autoreload

# Standard Library imports
import os
from pathlib import Path
from dataclasses import dataclass

# External imports
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import segmentation_models_pytorch as smp
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Local imports
from semantic_segmentation.train import main
from semantic_segmentation.model import make_deeplabv3_resnet101
from semantic_segmentation.loss import CombinedLoss, SymmetricUnifiedFocalLoss
from semantic_segmentation.metrics import DiceScore
from semantic_segmentation.scheduler import get_scheduler
from semantic_segmentation.dataset import SemanticSegmentationDataset
from semantic_segmentation.transforms import get_transforms
from semantic_segmentation.visualization import bar_plot, draw_batch
from semantic_segmentation.visualization import plot_loss_and_score, plot_score_per_class
from semantic_segmentation.visualization import visualize_classes, draw_predictions
from semantic_segmentation.visualization import visualize_rle_encoding_decoding
from semantic_segmentation.utils import LABELS_NAMES_MAP
from semantic_segmentation.utils import extract_and_onehot_encode_classes_from_multilabel_masks
from semantic_segmentation.utils import count_pixels_per_class
from semantic_segmentation.utils import load_checkpoint
from semantic_segmentation.utils import create_submission
from semantic_segmentation.utils import inference


plt.style.use('bmh')

<h2>Configuration</h2>

In [5]:
# class DataConfiguration:
#     # Cloud
#     #INPUT_PATH = "/workspace/input"
#     #DATA_PATH = "/workspace/input/data"
#     #OUTPUT_PATH = "/workspace/output"

#     # Local
#     INPUT_PATH = r"C:\Users\Manuel\Desktop\Documentos\1.PROJECTS\COMPUTER VISION\DLPT-semantic-segmentation\input"
#     DATA_PATH = r"F:\DATASETS\opencv-pytorch-segmentation-project"
#     OUTPUT_PATH = r"C:\Users\Manuel\Desktop\Documentos\1.PROJECTS\COMPUTER VISION\DLPT-semantic-segmentation\output"
    
#     # Kaggle
#     # INPUT_PATH = "/kaggle/input/"
#     # DATA_PATH = "/kaggle/input/opencv-pytorch-segmentation-project/"
#     # OUTPUT_PATH = "/kaggle/working"
    
#     # Data
#     TRAIN_SPLIT = 0.8
#     # This is the "background" class. It's NOT a 13th class, it is one of the 12 classes provided by the problem.
#     # In this problem, there are 11 classes of interest and a "background" class.
#     # In the case of semantic segmentation, the "background" class typically refers to the class representing regions of
#     # the image that do not belong to any of the predefined classes of interest. 
#     MASK_FILL_VALUE = 0
#     NUM_CLASSES = 12


@dataclass(frozen=True)
class Config:
    LOAD_MODEL_NAME = "deeplabv3_best_model.pt"

    # Cloud
    # INPUT_PATH = "/workspace/input"
    # DATA_PATH = "/workspace/input/data"
    # OUTPUT_PATH = "/workspace/output"

    # Local
    # INPUT_PATH = r"C:\Users\Manuel\Desktop"
    # DATA_PATH = r"F:\DATASETS\opencv-pytorch-segmentation-project"
    # OUTPUT_PATH = r"C:\Users\Manuel\Desktop\Documentos\1.PROJECTS\COMPUTER VISION\DLPT-semantic-segmentation\output"
    
    # Kaggle
    INPUT_PATH = "/kaggle/input/"
    DATA_PATH = "/kaggle/input/opencv-pytorch-segmentation-project/"
    OUTPUT_PATH = "/kaggle/working"
    
    # Data
    TRAIN_SPLIT = 0.8
    # This is the "background" class. It's NOT a 13th class, it is one of the 12 classes provided by the problem.
    # In this problem, there are 11 classes of interest and a "background" class.
    # In the case of semantic segmentation, the "background" class typically refers to the class representing regions of
    # the image that do not belong to any of the predefined classes of interest. 
    MASK_FILL_VALUE = 0
    NUM_CLASSES = 12

    # Training
    BATCH_SIZE = 2 if torch.cuda.is_available() else 2
    STARTING_EPOCH = 1  # one-indexed
    EPOCHS = 20
    SCHEDULER = None
    LR = 1e-3  # Used when LR scheduler is None or "constant"
    MAX_LR = 1e-3  # Used when the LR scheduler is "onecycle" or "cosine"
    MIN_LR = 1e-5  # Used when the LR scheduler is "cosine"
    # MOMENTUM = 0.937 # SGD momentum/Adam beta1, from Yolo v5
    # WEIGHT_DECAY = 0.0001 # Yolo v5 smart optimizer weight decay value
    GRADIENT_ACCUMULATION_STEPS = 4
    USE_AUX = True  # Toggle to use the auxiliary loss in DeepLabV3
    # UNFREEZE_BACKBONE_LAYERS = []
    # WEIGHT_LOSS_FUN1 = 1
    # WEIGHT_LOSS_FUN2 = 1

    # CROP_PATCHES = True
    RESIZE_WIDTH = None
    RESIZE_HEIGHT = None
    ORIGINAL_WIDTH = 1280
    ORIGINAL_HEIGHT = 720

    # Infrastructure
    NUM_WORKERS = 4  # os.cpu_count() # There are 4 CPUs in Kaggle
    DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # Automatic Mixed Precision
    USE_AMP = True  

    SUBMISSION_FILENAME = "submission.csv"

config = Config()

In [6]:
train_csv_path = Path(config.DATA_PATH) / "train.csv"
test_csv_path = Path(config.DATA_PATH) / "test.csv"

train_original_ids = pd.read_csv(train_csv_path).ImageID
test_ids = pd.read_csv(test_csv_path).ImageID

# <font style="color:green">1. Data Exploration</font>

## <font style="color:green">1.1. Dataset Class [7 Points]</font>

Split the training data into new training and validation sets, ensuring both have equivalent class proportions.

In [ ]:
original_dataset = SemanticSegmentationDataset(
    config.DATA_PATH, "imgs/imgs", "masks/masks", train_original_ids
)

X = train_original_ids
y = extract_and_onehot_encode_classes_from_multilabel_masks(original_dataset, config.NUM_CLASSES)

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=(1-config.TRAIN_SPLIT), random_state=0)
for train_index, val_index in msss.split(X, y):
    train_ids, valid_ids = X[train_index], X[val_index]

print(f"The new split has {len(train_ids)} train images and {len(valid_ids)} validation images.")

The train and validation splits have an equivalent proportion of images per class:

In [ ]:
classes_names = LABELS_NAMES_MAP.values()

normalized_image_counts_per_class = y.sum(axis=0) / y.sum()
bar_plot(classes_names, normalized_image_counts_per_class, title="Original training set", figsize=(12,2))

normalized_image_counts_per_class = y[train_index].sum(axis=0) / y[train_index].sum()
bar_plot(classes_names, normalized_image_counts_per_class, title="New training set", figsize=(12,2))

normalized_image_counts_per_class = y[val_index].sum(axis=0) / y[val_index].sum()
bar_plot(classes_names, normalized_image_counts_per_class, title="New validation set", figsize=(12,2))

In [ ]:
%autoreload 2

train_transforms, valid_transforms, test_transforms = get_transforms(
    config.MASK_FILL_VALUE
)

train_dataset = SemanticSegmentationDataset(
    config.DATA_PATH,
    "imgs/imgs",
    "masks/masks",
    train_ids.tolist(),
    transforms=train_transforms,
    target_height=config.RESIZE_HEIGHT,
    target_width=config.RESIZE_WIDTH
    
)

valid_dataset = SemanticSegmentationDataset(
    config.DATA_PATH,
    "imgs/imgs",
    "masks/masks",
    valid_ids.tolist(),
    transforms=valid_transforms,
    target_height=config.RESIZE_HEIGHT,
    target_width=config.RESIZE_WIDTH
)

test_dataset = SemanticSegmentationDataset(
    config.DATA_PATH,
    "imgs/imgs",
    "masks/masks",
    test_ids.tolist(),
    transforms=test_transforms,
    target_height=None,
    target_width=None
)

# Reason for drop_last: https://discuss.pytorch.org/t/error-expected-more-than-1-value-per-channel-when-training/26274/5
# Most likely you have a nn.BatchNorm layer somewhere in your model, which expects more then 1 value to calculate the running mean and std of the current batch.
train_dataloader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True,
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True,
)

print(f"Length train dataloader: ", len(train_dataloader))
print(f"Length valid dataloader: ", len(valid_dataloader))
print(f"Length test dataset: ", len(test_dataset))

## <font style="color:green">1.2. Visualize dataset [3 Points]</font>

In [ ]:
draw_batch(train_dataset, n_samples=10)

In [ ]:
draw_batch(valid_dataset, n_samples=10)

### Visualize each class

In [ ]:
visualize_classes(train_dataset, config.NUM_CLASSES)

### Count of pixels per class

In [ ]:
x = list(LABELS_NAMES_MAP.values())
y = count_pixels_per_class(train_original_ids, config.DATA_PATH, config.NUM_CLASSES)
bar_plot(x, y, xlabel="Class", ylabel="Pixel Count", title="Pixels Count per Class - All training data", figsize=(10,4))

# <font style="color:green">2. Evaluation Metrics [10 Points]</font>

<p>This competition is evaluated on the mean <a href='https://en.wikipedia.org/wiki/Sørensen–Dice_coefficient'>Dice coefficient</a
>. The Dice coefficient can be used to compare the pixel-wise agreement between a predicted segmentation and its corresponding ground truth. The formula is given by: </p>

<p>$$DSC =  \frac{2 |X \cap Y|}{|X|+ |Y|}$$
$$ \small \mathrm{where}\ X = Predicted\ Set\ of\ Pixels,\ \ Y = Ground\ Truth $$ </p>
<p>The Dice coefficient is defined to be 1 when both X and Y are empty.</p>

In [14]:
%autoreload 2
scorer = DiceScore(num_classes=config.NUM_CLASSES, ignore_index=None).to(config.DEVICE)

# <font style="color:green">3. Model [10 Points]</font>

#### Model

In [ ]:
model = make_deeplabv3_resnet101(config.NUM_CLASSES)
# unfreeze_deeplabv3_resnet101(model, config.UNFREEZE_BACKBONE_LAYERS)

#### Optimizer

In [16]:
optimizer = torch.optim.Adam(model.parameters(), lr=config.LR, amsgrad=True, fused=True)
# optimizer = torch.optim.SGD(model.parameters(), lr=config.LR, momentum=config.MOMENTUM, nesterov=False)
# optimizer = smart_optimizer(model, "RMSProp", lr=config.LR, momentum=config.MOMENTUM, decay=config.WEIGHT_DECAY)
# optimizer = smart_optimizer(model, "SGD", lr=config.MAX_LR, momentum=config.MOMENTUM, decay=config.WEIGHT_DECAY)

#### Scheduler

In [ ]:
steps_per_epoch = int(len(train_dataloader) / config.GRADIENT_ACCUMULATION_STEPS)
total_steps = int(steps_per_epoch * config.EPOCHS)
scheduler = get_scheduler(config.SCHEDULER, optimizer, total_steps, max_lr=config.MAX_LR, min_lr=config.MIN_LR)

In [18]:
scaler = torch.amp.grad_scaler.GradScaler("cuda", enabled=config.USE_AMP)

#### Load model and optimizer weights if available

In [ ]:
def optimizer_to(optim, device):
    """
    From: https://github.com/pytorch/pytorch/issues/8741#issuecomment-402129385
    """
    for param in optim.state.values():
        # Not sure there are any global tensors in the state dict
        if isinstance(param, torch.Tensor):
            param.data = param.data.to(device)
            if param._grad is not None:
                param._grad.data = param._grad.data.to(device)
        elif isinstance(param, dict):
            for subparam in param.values():
                if isinstance(subparam, torch.Tensor):
                    subparam.data = subparam.data.to(device)
                    if subparam._grad is not None:
                        subparam._grad.data = subparam._grad.data.to(device)
load_checkpoint(model, optimizer, config.INPUT_PATH, config.LOAD_MODEL_NAME, scaler)
model.to(config.DEVICE)
optimizer_to(optimizer, config.DEVICE)
print(optimizer)

#### Loss function

In [20]:
%autoreload 2

# Receives logits
# loss_fun1 = smp.losses.DiceLoss(mode="multiclass", from_logits=True).to(config.DEVICE)

# Receives logits
# loss_fun2 = smp.losses.FocalLoss("multiclass", normalized=False, reduction='mean').to(config.DEVICE)

# loss_fun = CombinedLoss(loss_fun1, loss_fun2, weight1=config.WEIGHT_LOSS_FUN1, weight2=config.WEIGHT_LOSS_FUN1).to(config.DEVICE)

# Focal TverskyLoss is defined by gamma=0.75
loss_fun = smp.losses.TverskyLoss(mode="multiclass", gamma=0.75)

# <font style="color:green">4. Train & Inference</font>

## <font style="color:green">4.1. Train [7 Points]</font>

In [ ]:
%autoreload 2

H = main(
    model,
    optimizer,
    scheduler,
    [loss_fun],
    scorer,
    scaler,
    train_dataloader,
    valid_dataloader,
    config.STARTING_EPOCH,
    config.EPOCHS,
    config.OUTPUT_PATH,
    config.GRADIENT_ACCUMULATION_STEPS,
    config.DEVICE,
    use_aux=config.USE_AUX,
    use_amp=config.USE_AMP
)

If the model is trained on downscaled images and masks, during validation its performance is reported on the downscaled 
versions. However, in real-world use, predictions will be evaluated against the original masks, not the downscaled ones.
That's why I need to report a final score on the validation set using upscaled predictions against the original masks.

Score on upscaled prediction and original mask:

In [ ]:
if config.RESIZE_HEIGHT is not None and config.RESIZE_WIDTH is not None:
    scores = inference(model, scorer, valid_ids, valid_transforms, config, upscale_prediction=True)  
    print(f"Mean validation score: {np.mean(scores)}")

Score on downscaled prediction and downscaled mask:

In [ ]:
if config.RESIZE_HEIGHT is not None and config.RESIZE_WIDTH is not None:
    scores = inference(model, scorer, valid_ids, valid_transforms, config, downscale_mask=True)  
    print(f"Mean validation score: {np.mean(scores)}")

In [ ]:
plot_loss_and_score(H)

In [ ]:
plot_score_per_class(H, config.NUM_CLASSES)

## <font style="color:green">4.2. Inference [3 Points]</font>

### Predictions on the validation set

In [ ]:
draw_predictions(model, valid_dataset, num_predictions=20, include_mask=True, device=config.DEVICE)

#### Verify RLE encoding
Visually verify that the RLE encoder works correctly by encoding and then decoding a mask

In [ ]:
image, mask = next(iter(valid_dataset))
visualize_rle_encoding_decoding(image, mask, config.NUM_CLASSES, titles=("Image", "Ground truth mask", "Encoded/decoded mask"))

### Predictions on the test set

In [ ]:
draw_predictions(model, test_dataset, num_predictions=20, include_mask=False, device=config.DEVICE)

# <font style="color:green">5. Prepare Submission CSV [10 Points]</font>

Format:
```
ImageID,EncodedPixels
01_0,1 1 5 1
01_1,2 3 8 1
02_0,1 1
02_1,3 1
03_0,1 1
03_1,4 5
etc.
```

In [ ]:
create_submission(model, test_ids, test_transforms, config)

In [ ]:
# TODO: save to a "run" directory
pd.read_csv(os.path.join(config.OUTPUT_PATH, config.SUBMISSION_FILENAME))

# <font style="color:green">6. Kaggle Profile Link [50 Points]</font>

Share your Kaggle profile link here with us so that we can give points for the competition score.

You should have a minimum IoU of `0.60` on the test data to get all points. If the IoU is less than `0.55`, you will not get any points for the section.

**You must have to submit `submission.csv` (prediction for images in `test.csv`) in `Submit Predictions` tab in Kaggle to get any evaluation in this section.**